# Archived Open NQ defense comparison on an A100

This notebook compares exact and lexical-family hints using the cached clean sample and the selected archive. The default archive contains selector `1+2` cases. The original promotion notebook configures selector `4` and performs selection and passage inference in its own pipeline. This replay uses the recorded passage responses and runs Mistral for the final answers.

In [ ]:
# Edit this configuration cell for your server and run.
from getpass import getuser
from pathlib import Path

PROJ = Path.home() / "rag-a100" / "project"
SHARED = PROJ
OUTPUT_FOLDER = getuser()

SOURCE_OWNER = "connor"
SOURCE_RUN_NAME = "table1-current_v2-open_nq-20261109T021501Z"
DATASET = "open_nq"
FROZEN_RUN_NAME = "promotion-live-open_nq-1+2-gold_alias_substitution-20261112T043046457994Z"

# Evaluate every question in the source manifest during the cache-only stage.
# The live limit applies only to questions with an archived result.
# In this notebook the switch enables final-answer replay of archived cases.
RUN_MISTRAL_ATTACK = True
MISTRAL_ATTACK_QUESTION_LIMIT = 100
MISTRAL_ATTACK_QUERY_INDICES = []  # An explicit list overrides the limit.

In [ ]:
import json, os, sys, subprocess
sys.dont_write_bytecode = True
os.chdir(PROJ)
sys.path.insert(0, str(PROJ))

def run_checked(args, **kwargs):
    completed = subprocess.run(
        [str(arg) for arg in args], check=False, **kwargs
    )
    if completed.returncode != 0:
        if getattr(completed, "stdout", None):
            print(completed.stdout, end="")
        if getattr(completed, "stderr", None):
            print(completed.stderr, end="", file=sys.stderr)
        raise RuntimeError(
            f"Command failed with exit status {completed.returncode}: "
            + " ".join(str(arg) for arg in args)
        )
    return completed

# The upstream 2024 dependencies require Python 3.11.
run_checked([sys.executable, "-m", "pip", "install", "-q", "uv==0.13.5"])
UV = [sys.executable, "-m", "uv"]
ENV_DIR = PROJ / ".robustrag-py311"
PYTHON = ENV_DIR / "bin/python"
if not PYTHON.exists():
    run_checked([*UV, "venv", "--python", "3.11", "--managed-python", ENV_DIR])
SPACY_MODEL = ("https://github.com/explosion/spacy-models/releases/download/"
               "en_core_web_sm-3.7.1/en_core_web_sm-3.7.1-py3-none-any.whl")
run_checked([*UV, "pip", "install", "--python", PYTHON,
             "torch==2.2.1", "numpy==1.26.4",
             "transformers==4.40.1", "tokenizers==0.19.1",
             "accelerate==0.29.3", "huggingface-hub==0.23.5",
             "sentencepiece==0.2.0", "joblib==1.3.2",
             "nltk==3.8.1", "spacy==3.7.4", "tqdm==4.66.1",
             "setuptools<81", SPACY_MODEL])
RUN_ENV = dict(os.environ, NLTK_DATA=str(PROJ / ".nltk-data"),
               PYTHONHASHSEED="0", PYTHONPATH=str(PROJ))
run_checked([PYTHON, "-m", "nltk.downloader", "-d",
             RUN_ENV["NLTK_DATA"], "stopwords"], env=RUN_ENV)
print("Experiment Python:", PYTHON)

In [ ]:
DATA_DIR = SHARED / "data" / "robustrag"
SOURCE_RUN = SHARED / "outputs" / SOURCE_OWNER / "robustrag-reproduction" / SOURCE_RUN_NAME
OUTPUT_ROOT = SHARED / "outputs" / OUTPUT_FOLDER / "lexical-family"
UPSTREAM_DIR = PROJ / "third_party" / "robustrag_upstream"
RUNNER_DIR = PROJ / ".robustrag-runner"

manifest = json.loads((SOURCE_RUN / "manifest.json").read_text())
if (manifest["status"], manifest["profile"], manifest["dataset"]) != ("complete", "current_v2", DATASET):
    raise ValueError(f"Expected a completed current_v2 {DATASET} source run in {SOURCE_RUN}")
print("Cached input:", SOURCE_RUN)
print("Output root:", OUTPUT_ROOT)
FROZEN_RUN = SHARED / "outputs" / SOURCE_OWNER / "promotion-attack" / FROZEN_RUN_NAME

## Prepare the vendored authors' code and verified data

The pinned upstream Python source is committed under `third_party/`, so this step makes no GitHub code download. Shared JSON datasets stay outside git and are downloaded only when missing, with SHA-256 verification.

In [ ]:
from reproduction.robustrag_setup import build_runner, ensure_datasets

build_runner(UPSTREAM_DIR, RUNNER_DIR)
ensure_datasets(DATA_DIR, (DATASET,))

preflight = run_checked(
    [PYTHON, "-c", "import spacy, torch, transformers, joblib; from nltk.corpus import stopwords; from src.models import create_model; assert 'the' in stopwords.words('english'); assert spacy.load('en_core_web_sm')('Paris')[0].pos_; print(torch.__version__, transformers.__version__)"],
    cwd=RUNNER_DIR, env=RUN_ENV, capture_output=True, text=True
)
print("Preflight:", preflight.stdout.strip())

## Cache-only hint selection across the source sample

In [ ]:
# Use the same pinned Python as preflight. Pass paths as individual arguments.
replay = """
import json, sys
from pathlib import Path
import spacy
from nltk.corpus import stopwords
from defense_selectors import LexicalFamilySelector
from promotion_attack import SpacyKeywordExtractor, aggregate_keyword_sets, load_upstream_question_records

source, dataset_file, runner = sys.argv[1:]
records, manifest = load_upstream_question_records(source, dataset_file, runner)
params = manifest["parameters"]
nlp = spacy.load("en_core_web_sm")
extractor = SpacyKeywordExtractor(nlp)
selector = LexicalFamilySelector(nlp)
stop = set(stopwords.words("english"))
rows = []
for record in records:
    votes = [extractor.extract(response) for response in record.responses]
    exact = aggregate_keyword_sets(votes, params["alpha"], params["beta"], stop)
    selected = selector.select(exact.retained_keywords, record.responses, votes)
    rows.append(dict(query_index=record.query_index, question=record.question,
                     exact=exact.retained_keywords, lexical_family=selected))
print(json.dumps(rows, ensure_ascii=False))
"""
result = run_checked(
    [PYTHON, "-c", replay, SOURCE_RUN, DATA_DIR / f"{DATASET}.json", RUNNER_DIR],
    cwd=PROJ, env=RUN_ENV, capture_output=True, text=True,
)
rows = json.loads(result.stdout.strip().splitlines()[-1])
print(f"Processed all {len(rows)} cached source questions.")

In [ ]:
from datetime import datetime, timezone
from reproduction.robustrag_setup import file_sha256

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
out = OUTPUT_ROOT / stamp
out.mkdir(parents=True, exist_ok=False)
(out / "hints.json").write_text(json.dumps(rows, indent=2, ensure_ascii=False))
run = {
    "kind": "clean_cached_hint_comparison", "hint_selector": "lexical_family",
    "source_run": str(SOURCE_RUN), "source_manifest_sha256": file_sha256(SOURCE_RUN / "manifest.json"),
    "selector_sha256": file_sha256(PROJ / "defense_selectors.py"),
    "dataset": DATASET, "parameters": manifest["parameters"], "source_sample_limit": manifest["sample_limit"],
    "query_indices": [row["query_index"] for row in rows], "sample_size": len(rows),
}
(out / "manifest.json").write_text(json.dumps(run, indent=2))
print("Download this directory:", out)

In [ ]:
changed = sum(row["exact"] != row["lexical_family"] for row in rows)
print(f"Clustering changed the displayed hints for {changed}/{len(rows)} source questions.")
for row in rows[:5]:
    print(f"Q{row['query_index']}: {len(row['exact'])} retained, {len(row['lexical_family'])} displayed")

if RUN_MISTRAL_ATTACK:
    from scripts.run_lexical_family_pilot import pilot_indices

    archive = json.loads((FROZEN_RUN / "manifest.json").read_text())
    archived = archive["query_indices"]
    planned = pilot_indices(archived, MISTRAL_ATTACK_QUESTION_LIMIT, MISTRAL_ATTACK_QUERY_INDICES or None)
    print(f"Live pilot: {len(planned)}/{len(archived)} archived questions from {len(rows)} source questions.")
    print("Frozen configuration:", archive["config"])
    print("Live query indices:", planned)

## Optional bounded live pilot on the A100

The pilot uses the archived cases in their saved order. The question limit is an upper bound. An explicit list selects those archived cases in the order supplied. Each archived candidate keeps its rank. Outputs include target and gold presence, strict targeted success, and success@1, @3 and @all across the archived candidate prefix. The model generates one answer per distinct prompt within that question.

In [ ]:
if RUN_MISTRAL_ATTACK:
    live_command = [
        PYTHON, "-u", "-m", "scripts.run_lexical_family_pilot",
        "--source-run", SOURCE_RUN, "--dataset-file", DATA_DIR / f"{DATASET}.json",
        "--runner", RUNNER_DIR, "--frozen-run", FROZEN_RUN, "--output-root", out,
    ]
    if MISTRAL_ATTACK_QUERY_INDICES:
        live_command.extend(["--query-indices", *map(str, MISTRAL_ATTACK_QUERY_INDICES)])
    else:
        live_command.extend(["--limit", str(MISTRAL_ATTACK_QUESTION_LIMIT)])
    run_checked(live_command, cwd=PROJ, env=RUN_ENV)
else:
    print("Cache-only run:", out)

In [ ]:
print("Download this run directory:", out)